In [1]:
import pandas as pd

# Extract
excel_path = "Coffee Shop Sales (2).xlsx"  # adjust path if needed

# Read the Transactions sheet
df = pd.read_excel(excel_path, sheet_name="Transactions")

# View the first few rows
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [3]:
# Transform
df['transaction_date'] = pd.to_datetime(df['transaction_date']).dt.date
df['transaction_time'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S', errors='coerce').dt.time
df['total_amount'] = df['transaction_qty'] * df['unit_price']

# Create dimension tables
dim_store = df[['store_id', 'store_location']].drop_duplicates().reset_index(drop=True)
dim_product = df[['product_id', 'product_category', 'product_type', 'product_detail']].drop_duplicates().reset_index(drop=True)

# Date dimension
dim_date = pd.DataFrame({
    'date_key': pd.to_datetime(df['transaction_date']),
})
dim_date['year'] = dim_date['date_key'].dt.year
dim_date['month'] = dim_date['date_key'].dt.month
dim_date['day'] = dim_date['date_key'].dt.day
dim_date['weekday_name'] = dim_date['date_key'].dt.day_name()
dim_date = dim_date.drop_duplicates().reset_index(drop=True)

# Time dimension
dim_time = pd.DataFrame({
    'time_key': pd.to_datetime(df['transaction_time'].astype(str), format='%H:%M:%S', errors='coerce')
})
dim_time['hour'] = dim_time['time_key'].dt.hour
dim_time['minute'] = dim_time['time_key'].dt.minute
dim_time['timeslot'] = dim_time['hour'].apply(
    lambda h: 'morning' if 5 <= h < 12 else
              'afternoon' if 12 <= h < 17 else
              'evening' if 17 <= h < 22 else 'night'
)
dim_time = dim_time.drop_duplicates().reset_index(drop=True)

# Fact table (transactions)
fact_transactions = df[['transaction_id', 'transaction_date', 'transaction_time',
                        'store_id', 'product_id', 'transaction_qty',
                        'unit_price', 'total_amount']].copy()

fact_transactions.head()

,transaction_id,transaction_date,transaction_time,store_id,product_id,transaction_qty,unit_price,total_amount
0,1,2023-01-01,07:06:11,5,32,2,3.0,6.0
1,2,2023-01-01,07:08:56,5,57,2,3.1,6.2
2,3,2023-01-01,07:14:04,5,59,2,4.5,9.0
3,4,2023-01-01,07:20:24,5,22,1,2.0,2.0
4,5,2023-01-01,07:22:41,5,57,2,3.1,6.2


In [4]:
import sqlalchemy

# Load
engine = sqlalchemy.create_engine("sqlite:///coffee_shop.db")

# Load tables
dim_store.to_sql("dim_store", engine, if_exists="replace", index=False)
dim_product.to_sql("dim_product", engine, if_exists="replace", index=False)
dim_date.to_sql("dim_date", engine, if_exists="replace", index=False)
dim_time.to_sql("dim_time", engine, if_exists="replace", index=False)
fact_transactions.to_sql("fact_transactions", engine, if_exists="replace", index=False)

print("✅ All tables loaded into SQLite successfully!")

✅ All tables loaded into SQLite successfully!


In [6]:
!git init

Initialized empty Git repository in C:/Users/Dell/Documents/Project Pipeline/.git/


In [7]:
!git checkout -b main

Switched to a new branch 'main'


In [8]:
!git checkout -b feature/etl-pipeline

Switched to a new branch 'feature/etl-pipeline'


In [ ]:
!git add .
!git commit -m "Add ETL pipeline notebook + SQLite DB"